<a href="https://colab.research.google.com/github/swagatabanerjee/QML-Course-Practice/blob/main/Quantum_Error_Correction_with_qiskit_qec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Experimental QEC dataset

This notebook uses Google's published surface-code memory data from a real quantum processor: [Zenodo record 13273331](https://zenodo.org/records/13273331), released under CC BY 4.0. `qiskit-qec` does not bundle this dataset. Download `google_105Q_surface_code_d3_d5_d7.zip` (about 5.7 GB), extract it under a `data` folder beside this notebook, and keep the extracted directory structure intact. The loader uses the distance-3, X-basis, 13-round sample at `data/google_105Q_surface_code_d3_d5_d7/d3_at_q4_5/X/r13/`.

The archive includes raw packed measurement records, detector events, and actual logical-observable flips. The notebook trains on detector events and predicts the actual logical flip. To keep IBM QPU runs manageable, it takes a balanced sample of up to 128 shots per class and reduces the detector vector to two features with PCA; reported accuracy is therefore for that balanced subset, not the experiment's natural logical-error rate.

In [1]:
# Install Qiskit, IBM Runtime, Stim (for the dataset's packed QEC data), and notebook dependencies
%pip install stim qiskit qiskit-aer qiskit-algorithms qiskit-ibm-runtime qiskit-machine-learning stim pylatexenc matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=2,
)
print(f"Selected QPU: {backend.name}")

pass_manager = generate_preset_pass_manager(
    backend=backend,
    optimization_level=1,
)
sampler = SamplerV2(mode=backend)
sampler.options.default_shots = 512

In [ ]:
import json
from pathlib import Path

import numpy as np
import stim
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit_algorithms.optimizers import COBYLA
from qiskit_machine_learning.algorithms import VQC

# Google's distance-3, X-basis, 13-round experimental surface-code sample.
DATA_DIR = Path("data/google_105Q_surface_code_d3_d5_d7/google_105Q_surface_code_d3_d5_d7/d3_at_q4_5/X/r13")
required_files = [
    DATA_DIR / "metadata.json",
    DATA_DIR / "circuit_ideal.stim",
    DATA_DIR / "detection_events.b8",
    DATA_DIR / "obs_flips_actual.b8",
]
missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Dataset files are missing. Download and extract the Google QEC archive "
        "as described in the notebook's Experimental QEC dataset section. Missing: "
        + ", ".join(str(path) for path in missing_files)
    )

metadata = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))
circuit = stim.Circuit.from_file(str(DATA_DIR / "circuit_ideal.stim"))
num_shots = metadata["shots"]
num_detectors = circuit.num_detectors
num_observables = circuit.num_observables
if num_observables != 1:
    raise ValueError(f"Expected one logical observable, found {num_observables}.")


def read_b8(path, shots, bits_per_shot):
    packed = np.fromfile(path, dtype=np.uint8)
    bytes_per_shot = (bits_per_shot + 7) // 8
    expected_size = shots * bytes_per_shot
    if packed.size != expected_size:
        raise ValueError(
            f"Unexpected b8 size for {path}: expected {expected_size} bytes "
            f"({shots} shots x {bytes_per_shot} bytes), got {packed.size}."
        )
    packed = packed.reshape(shots, bytes_per_shot)
    return np.unpackbits(packed, axis=1, bitorder="little")[:, :bits_per_shot]


# Detector flips are the inputs; the measured logical-observable flip is the target.
X = read_b8(DATA_DIR / "detection_events.b8", num_shots, num_detectors).astype(np.float32)
y = read_b8(DATA_DIR / "obs_flips_actual.b8", num_shots, num_observables)[:, 0].astype(np.int64)
if not np.array_equal(np.unique(y), np.array([0, 1])):
    raise ValueError("The selected dataset sample must contain both logical-flip classes.")

# Use a balanced subset to control QPU runtime and prevent the rare class being ignored.
max_per_class = 128
class_indices = [np.flatnonzero(y == label) for label in (0, 1)]
samples_per_class = min(max_per_class, *(len(indices) for indices in class_indices))
if samples_per_class < 4:
    raise ValueError("Too few samples in one logical-flip class for a stratified split.")

rng = np.random.default_rng(42)
sample_indices = np.concatenate([
    rng.choice(indices, size=samples_per_class, replace=False)
    for indices in class_indices
])
rng.shuffle(sample_indices)
X_sample = X[sample_indices]
y_sample = y[sample_indices]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_sample,
    y_sample,
    test_size=0.25,
    random_state=42,
    stratify=y_sample,
)

# The VQC uses two qubits; project the 104 detector bits to two features using train-only fits.
standardizer = StandardScaler()
pca = PCA(n_components=2, random_state=42)
X_train = pca.fit_transform(standardizer.fit_transform(X_train_raw))
X_test = pca.transform(standardizer.transform(X_test_raw))
angle_scaler = MinMaxScaler(feature_range=(-1, 1))
X_train = angle_scaler.fit_transform(X_train)
X_test = angle_scaler.transform(X_test)

print(f"Dataset: distance {metadata['distance']}, {metadata['rounds']} rounds, basis {metadata['basis']}")
print(f"Raw records: {num_shots:,} shots, {num_detectors} detectors, {num_observables} observable")
print(f"Full logical-flip counts: {np.bincount(y, minlength=2).tolist()}")
print(f"Balanced subset: {samples_per_class} samples per class")
print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

In [ ]:
from sklearn.svm import SVC

# 1. Initialize and train the classical Support Vector Machine (RBF kernel)
svm_model = SVC(kernel='rbf', C=1.0, random_state=42)
svm_model.fit(X_train, y_train)

# 2. Evaluate performance
svm_train_score = svm_model.score(X_train, y_train)
svm_test_score = svm_model.score(X_test, y_test)

print(f"Classical SVM Training Accuracy: {svm_train_score * 100:.2f}%")
print(f"Classical SVM Testing Accuracy:  {svm_test_score * 100:.2f}%")


In [ ]:
from sklearn.svm import SVC

# 1. Initialize and train the classical Support Vector Machine (RBF kernel)
svm_model = SVC(kernel='rbf', C=1.0, random_state=42)
svm_model.fit(X_train, y_train)

# 2. Evaluate performance
svm_train_score = svm_model.score(X_train, y_train)
svm_test_score = svm_model.score(X_test, y_test)

print(f"Classical SVM Training Accuracy: {svm_train_score * 100:.2f}%")
print(f"Classical SVM Testing Accuracy:  {svm_test_score * 100:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

if 'objective_func_vals' in globals() and len(objective_func_vals) > 0:
    plt.figure(figsize=(8, 5))
    plt.title("VQC Training Loss Curve")
    plt.xlabel("Iteration")
    plt.ylabel("Objective Function Value")
    plt.plot(range(len(objective_func_vals)), objective_func_vals, label='COBYLA loss', color='tab:blue')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.show()
else:
    print("No objective function values were recorded in 'objective_func_vals'. Please re-run the VQC training cell to populate the values.")

In [ ]:
# Combine feature map and ansatz to visualize the full VQC circuit structure
vqc_circuit = feature_map.compose(ansatz)

# Draw the circuit using matplotlib representation
vqc_circuit.draw(output='mpl')